# Pré-processamento pro vosviewer

# Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install nltk unidecode -q

In [4]:
import pandas as pd
import ast
import re
import unicodedata
from unidecode import unidecode
from collections import Counter, defaultdict
import plotly.express as px

In [5]:
path = "/content/drive/MyDrive/TJMA LACA/artigos_jmoe1.csv"
df = pd.read_csv(path)

In [6]:
df.head()

,_id,ano,volume,numero,titulo,autores,palavras_chave,abstract,referencias,link,...,Citações,pre_keywords,university,pais,location,cidade,estado,texto_documento_preprocessed,universities_translated,universities
0,68a8cad06a09251fb721014d,2025,24,1,A Flexible UHF RFID Respiratory Sensor,"['Araujo, Jéssyca I. L.', 'Serres, Georgina K....",Index Terms\\\\nFlexible tag; Metamaterial-ins...,"In this study, we developed a Radio Frequency ...","[1] M. A. Cretikos, R. Bellomo, K. Hillman, J....",https://www.scielo.br/j/jmoea/a/XHgN4VqNZCD5cC...,...,0,"flexible tag , metamaterial inspired , respira...","['Federal University of Campina Grande', 'Fede...","['Brazil', 'Brazil', 'Italy', 'Brazil', 'Brazil']","['Campina Grande, Paraíba, Brazil', 'Campina G...","[Campina Grande, Pisa, Campina Grande, Campina...","[Paraíba, Paraíba, Pisa, Paraíba, Paraíba]",federal university of campina grande; federal ...,Universidade Federal de Campina Grande; Univer...,"['universidade federal de campina grande', 'un..."
1,68a8cad36a09251fb721014e,2025,24,1,Proposition for Path Loss Prediction Models wi...,"['Jorge Júnior, Evandro M.', 'Veiga, Antônio C...",Index Terms\\\\nDifferential Evolution; Geneti...,"In this paper, it is presented a general optim...","[1] N. Faruk, S. I. Popoola, N. T. Surajudeen-...",https://www.scielo.br/j/jmoea/a/XFZhWqy65ZzgzZ...,...,0,"differential evolution , genetic algorithm , p...","['Universidade Federal de Uberlândia', 'Univer...","['Brazil', 'Brazil', 'Brazil']","['Uberlândia, Minas Gerais, Brazil', 'Uberlând...","[Uberlândia, Uberlândia, Uberlândia]","[Minas Gerais, Minas Gerais, Minas Gerais]",universidade federal de uberlandia; universida...,Universidade Federal de Uberlandia; Universida...,"['universidade federal de uberlandia', 'univer..."
2,68a8cad66a09251fb721014f,2025,24,1,A 1D-FDM Transmission Line Model for Partial D...,"['Oliveira, Rodrigo M. S. de', 'Lopes, Nathan ...",Index Terms\\\\nEffective Electrical Conductiv...,"In this paper, we present a novel numerical mo...","[1] A. Haddad and D. F. Warne, Eds., Advances ...",https://www.scielo.br/j/jmoea/a/WQxvPr4GjWVKVw...,...,0,"effective electrical conductivity , finite dif...","['Federal University of Pará', 'UFPA-ITEC', 'E...","['Brazil', 'Brazil']","['Belém, Brazil', 'Belém, Brazil']","[Belém, Belém]","[Brazil, Brazil]",federal university of para; ufpa itec; eletrob...,Universidade Federal do Pará; Universidade Fed...,"['universidade federal do para', 'universidade..."
3,68a8cad86a09251fb7210150,2025,24,1,Assessment of Planar Transmission Line with St...,"['Fonseca, Daniel A. B.', 'Costa, Luís G. da S...",Index Terms\\\\nPlanar transmission line; powe...,This study investigates the effectiveness of e...,"[1] D. K. Mahanta and O. Andrew, “Transformer ...",https://www.scielo.br/j/jmoea/a/vr67fPZqgtyFVb...,...,0,"planar transmission line , power transformer ,...","['Universidade Federal de Juiz de Fora', 'Univ...","['Brazil', 'Brazil', 'Brazil', 'Brazil', 'Braz...","['Juiz de Fora, Minas Gerais, Brazil', 'Juiz d...","[Juiz de Fora, Juiz de Fora, Juiz de Fora, Jui...","[Minas Gerais, Minas Gerais, Minas Gerais, Min...",universidade federal de juiz de fora; universi...,Universidade Federal de Juiz de Fora; Universi...,"['universidade federal de juiz de fora', 'univ..."
4,68a8cada6a09251fb7210151,2025,24,1,Space-fed Array Variations of E-shape and U-sl...,"['Deshmukh, Amit A.', 'Parvez, Adil', 'Chavali...",Index Terms\\\\nBroadband High gain microstrip...,Space-fed array comprising of E-shape or U-slo...,"[1] G. Kumar, K. P. Ray, Broadband Microstrip ...",https://www.scielo.br/j/jmoea/a/Nkr93SPpvfsMLP...,...,0,"broadband high gain microstrip antenna , shap...",['Dwarkadas Jivanlal Sanghvi College of Engine...,"['India', 'India', 'India', 'India']","['Mumbai, India', 'Mumbai, India', 'Mumbai, In...","[Mumbai, Mumbai, Mumbai, Mumbai]","[India, India, India, India]",dwarkadas jivanlal sanghvi college of engineer...,Faculdade de Engenharia Dwarkadas Jivanlal San...,['fa

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570 entries, 0 to 569
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   _id                           570 non-null    object
 1   ano                           570 non-null    int64 
 2   volume                        570 non-null    int64 
 3   numero                        570 non-null    int64 
 4   titulo                        565 non-null    object
 5   autores                       570 non-null    object
 6   palavras_chave                460 non-null    object
 7   abstract                      461 non-null    object
 8   referencias                   458 non-null    object
 9   link                          570 non-null    object
 10  universidade                  490 non-null    object
 11  Citações                      570 non-null    object
 12  pre_keywords                  460 non-null    object
 13  university          

In [8]:
#limpeza geral

def clean_list(x):
    if pd.isna(x):
        return ""
    try:
        lst = ast.literal_eval(x)
        return "; ".join(str(i).strip() for i in lst)
    except:
        return str(x)

def clean_keywords(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"Index Terms", "", x, flags=re.I)

    x = x.replace(r"\n", " ")
    x = re.sub(r"\\n", " ", x)
    x = re.sub(r"\bn(?=[A-Z])", "", x)

    x = re.sub(r"[\n\t\\]+", " ", x)
    x = re.sub(r"\s+", " ", x)

    parts = [p.strip(" .") for p in x.split(';') if p.strip(" .")]
    return "; ".join(parts)

In [9]:
# Padronizar autores e keywords
# Aplica as limpezas
df["authors"] = df["autores"].apply(clean_list)
df["authors"] = df["authors"].apply(lambda x: unidecode(x) if isinstance(x, str) else x)
df["affiliations"] = df["university"].apply(clean_list)
df["Author Keywords"] = df["palavras_chave"].apply(clean_keywords)


# Lógica de Plurais nas Palavras-Chave
todas_kws = []
for row in df["Author Keywords"].dropna():
    for kw in row.split(';'):
        kw = kw.strip()
        if kw: todas_kws.append(kw)

kw_case_freq = defaultdict(Counter)
for kw in todas_kws:
    kw_case_freq[kw.lower()][kw] += 1

best_case_kw = {k: v.most_common(1)[0][0] for k, v in kw_case_freq.items()}
lowercased_kws = set(best_case_kw.keys())
plural_to_singular = {}

for kw in lowercased_kws:
    if kw.endswith('s') and not kw.endswith('ss') and not kw.endswith('is'):
        if kw[:-1] in lowercased_kws:
            plural_to_singular[kw] = kw[:-1]
        elif kw.endswith('ies') and kw[:-3] + 'y' in lowercased_kws:
            plural_to_singular[kw] = kw[:-3] + 'y'
        elif kw.endswith('es') and kw[:-2] in lowercased_kws:
            plural_to_singular[kw] = kw[:-2]

def normalize_plurals_and_case(text):
    if not text: return text
    parts = [p.strip() for p in text.split(';') if p.strip()]
    final_parts = []
    for p in parts:
        low = p.lower()
        low = plural_to_singular.get(low, low)
        final_parts.append(best_case_kw.get(low, p.title()))
    final_parts = list(dict.fromkeys(final_parts))
    return "; ".join(final_parts)

In [10]:
df["Author Keywords"] = df["Author Keywords"].apply(normalize_plurals_and_case)

In [11]:
def parse_geo_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            res = ast.literal_eval(x)
            if isinstance(res, list): return res
        except:
            x = x.strip("[]")
            return [e.strip() for e in x.split(",") if e.strip()]
    return []

df["pais"] = df["pais"].apply(parse_geo_list)
df["estado"] = df["estado"].apply(parse_geo_list)
df["cidade"] = df["cidade"].apply(parse_geo_list)

In [12]:
# Mapeamento do Estado -> País (Para preencher países vazios)
map_estado_pais = {
    "Minas Gerais": "Brazil", "Goias": "Brazil", "Goiás": "Brazil", "Ceará": "Brazil",
    "Paraná": "Brazil", "Parana": "Brazil", "Pará": "Brazil", "Para": "Brazil",
    "Paraíba": "Brazil", "Paraiba": "Brazil", "Rio Grande do Norte": "Brazil",
    "Rio Grande do Sul": "Brazil", "Bahia": "Brazil", "Santa Catarina": "Brazil",
    "Distrito Federal": "Brazil", "Federal District": "Brazil", "Espirito Santo": "Brazil",
    "Vitória-Espirito Santo": "Brazil", "São Paulo": "Brazil", "Sao Paulo": "Brazil",
    "Mato Grosso do Sul": "Brazil", "Pernambuco": "Brazil", "TO": "Brazil",
    "Telangana": "India", "Telengana": "India", "Odisha": "India", "Rajasthan": "India",
    "Andhra Paranaadesh": "India", "Uttar Paranaadesh": "India", "Punjab": "India",
    "Jharkhand": "India", "West Bengal": "India", "Maharashtra": "India", "Karnataka": "India",
    "U.P.": "India", "Ranchi": "India", "Liaoning 125105": "China", "Liaoning116026": "China",
    "Nanjing": "China", "Xian": "China", "Johor": "Malaysia", "Kuala Lumpur": "Malaysia",
    "Lagos": "Nigeria", "Quebec": "Canada", "Pisa": "Italy", "Rome": "Italy", "KSA": "Saudi Arabia",
    "Algérie": "Algeria", "Algerie": "Algeria", "Algeria": "Algeria", "M’sila 28000": "Algeria",
    "Iran": "Iran", "Morocco": "Morocco", "Ghana": "Ghana", "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "Portugal": "Portugal", "Sweden": "Sweden", "Türkiye": "Turkey", "Taiwan": "Taiwan",
    "Ukraine": "Ukraine", "Viet Nam": "Vietnam", "Perú": "Peru", "Syria": "Syria", "Iraq": "Iraq", "Mali": "Mali"
}

def substituir_por_dicionario(row):
    paises = row["pais"]
    estados = row["estado"]

    # Caso 1: Se a lista de países está vazia, mas há estados
    if len(paises) == 0 and len(estados) > 0:
        return [map_estado_pais.get(e, "") for e in estados]

    # Caso 2: Listas do mesmo tamanho
    if len(estados) == len(paises):
        novo = []
        for p, e in zip(paises, estados):
            if (not p or p == "") and e in map_estado_pais:
                novo.append(map_estado_pais[e])
            else:
                novo.append(p)
        return novo

    return paises

In [13]:
df["pais"] = df.apply(substituir_por_dicionario, axis=1)

In [14]:
df["countries"] = df["pais"].apply(lambda x: "; ".join(str(i).strip() for i in x if i) if isinstance(x, list) else str(x))

In [15]:
vos = df[[
    "authors",
    "titulo",
    "ano",
    "affiliations",
    "countries",
    "Author Keywords",
    "Citações"
]].copy()

In [16]:
vos.columns = [
    "Authors", "Title", "Year", "Affiliations", "Country", "Author Keywords", "Cited by"
]

In [17]:
vos.head()

,Authors,Title,Year,Affiliations,Country,Author Keywords,Cited by
0,"Araujo, Jessyca I. L.; Serres, Georgina K. F.;...",A Flexible UHF RFID Respiratory Sensor,2025,Federal University of Campina Grande; Federal ...,Brazil; Brazil; Italy; Brazil; Brazil,Flexible tag; Metamaterial-inspired; Respirato...,0
1,"Jorge Junior, Evandro M.; Veiga, Antonio C. P.",Proposition for Path Loss Prediction Models wi...,2025,Universidade Federal de Uberlândia; Universida...,Brazil; Brazil; Brazil,Differential Evolution; Genetic Algorithm; pat...,0
2,"Oliveira, Rodrigo M. S. de; Lopes, Nathan M.; ...",A 1D-FDM Transmission Line Model for Partial D...,2025,Federal University of Pará; UFPA-ITEC; Eletrob...,Brazil; Brazil,Effective Electrical Conductivity; Finite-Diff...,0
3,"Fonseca, Daniel A. B.; Costa, Luis G. da S.; C...",Assessment of Planar Transmission Line with St...,2025,Universidade Federal de Juiz de Fora; Universi...,Brazil; Brazil; Brazil; Brazil; Brazil,Planar transmission line; power transformer; M...,0
4,"Deshmukh, Amit A.; Parvez, Adil; Chavali, Venk...",Space-fed Array Variations of E-shape and U-sl...,2025,Dwarkadas Jivanlal Sanghvi College of Engineer...,India; India; India; India,Broadband High gain microstrip antenna; E-shap...,0


In [18]:
vos.to_csv("/content/drive/MyDrive/TJMA LACA/vosviewer_ready.csv", index=False)

In [19]:
# ANÁLISE GEOGRÁFICA E PLOTLY MAP

In [20]:
df_paises = df.explode("pais")
df_paises = df_paises[df_paises["pais"].notna() & (df_paises["pais"] != "")]
df_paises["pais"] = df_paises["pais"].astype(str).str.strip()

In [21]:
# Padronização final de nomenclatura global
mapa_paises = {
    "Brasil": "Brazil", "Brazi": "Brazil", "INDIA": "India", "Indi": "India",
    "P. R. China": "China", "P.R.China": "China", "ROC": "Taiwan", "Roc": "Taiwan",
    "UK": "United Kingdom", "Uk": "United Kingdom", "KSA": "Saudi Arabia", "Ksa": "Saudi Arabia",
    "Türkiye": "Turkey", "Perú": "Peru", "Algérie": "Algeria", "Algerie": "Algeria",
    "Viet Nam": "Vietnam", "Morocco ": "Morocco"
}

In [22]:
df_paises["pais"] = df_paises["pais"].replace(mapa_paises)

In [23]:
# removemos duplicatas para não contar o mesmo país 2x no mesmo artigo
df_paises = df_paises.drop_duplicates(subset=["_id", "pais"])

In [24]:
contagem = df_paises["pais"].value_counts().reset_index()
contagem.columns = ["pais", "publicacoes"]

In [25]:
# Contagem
contagem["publicacoes"] = pd.to_numeric(contagem["publicacoes"], errors="coerce")

In [26]:
print(contagem["publicacoes"].describe())
print(contagem.sort_values("publicacoes", ascending=False).head(20))

count     33.000000
mean      13.727273
std       51.652851
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max      297.000000
Name: publicacoes, dtype: float64
            pais  publicacoes
0         Brazil          297
1          India           48
2          China           20
3        Algeria           17
4           Iran           14
5        Morocco            9
6        Nigeria            5
7        Ukraine            4
8         Canada            3
9       Portugal            3
10         Italy            3
11      Malaysia            3
12  Saudi Arabia            2
13      Colombia            2
14         Egypt            2
15        Turkey            2
16          Iraq            2
17        Taiwan            2
18        France            1
19       Vietnam            1


In [ ]:
# !pip uninstall -y kaleido
# !pip install kaleido==0.2.1

In [ ]:
import plotly
import kaleido

print("plotly:", plotly.__version__)
print("kaleido:", kaleido.__version__)

In [ ]:
import kaleido
print("kaleido ok")

In [42]:
def faixa_pub(x):
    if x == 1:
        return "1"
    elif x <= 3:
        return "2–3"
    elif x <= 9:
        return "4–9"
    elif x <= 49:
        return "10–49"
    else:
        return "50+"

contagem["faixa"] = contagem["publicacoes"].apply(faixa_pub)

fig = px.choropleth(
    contagem,
    locations="pais",
    locationmode="country names",
    color="faixa",
    category_orders={"faixa": ["1", "2–3", "4–9", "10–49", "50+"]},
    color_discrete_map={
        "1": "#f7f7f7",
        "2–3": "#d9d9d9",
        "4–9": "#9ecae1",
        "10–49": "#3182bd",
        "50+": "#08519c"
    },
    title="Global Distribution of Publications",
    hover_name="pais",
    hover_data={"publicacoes": True, "faixa": False}
)

fig.update_layout(
    title={
        "text": "Global Distribution of Publications",
        "x": 0.5,
        "xanchor": "center",
        "font": dict(size=20)
    },
    font=dict(family="Times New Roman", size=16),
    legend=dict(
        title=dict(
            text="Publications",
            font=dict(size=18)
        ),
        font=dict(size=16),
        x=0.84,
        y=0.90,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.96)",
        bordercolor="rgba(80,80,80,0.8)",
        borderwidth=1
    ),
    margin=dict(l=10, r=30, t=70, b=10),
    width=1600,
    height=900
)

fig.update_geos(
    showframe=False,
    showcoastlines=False,
    projection_type="natural earth"
)

fig.show()

In [43]:
fig.write_image(
    "/content/drive/MyDrive/TJMA LACA/mapa_global_publicacoes.png",
    width=1600,
    height=900,
    scale=4
)

In [33]:
# Gráfico de Barras Top 10
top10 = contagem.nlargest(10, "publicacoes")
fig_bar = px.bar(
    top10.sort_values("publicacoes"),
    x="publicacoes",
    y="pais",
    orientation="h",
    title="Top 10 Contributing Countries",
    labels={"publicacoes": "Number of Publications", "pais": "Country"}
)
fig_bar.update_layout(font=dict(family="Times New Roman", size=14),
                      yaxis=dict(categoryorder="total ascending"))
fig_bar.show()

# # Salvar Imagens
# fig.write_image("mapa_global_publicacoes.png", scale=4)
# fig_bar.write_image("top10_paises.png", scale=4)